In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


## Support vector Regression Implementation

In [5]:
df = sns.load_dataset('tips')

In [7]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [8]:
# Independent and Dependent features
X = df[['tip','sex','smoker','day','time','size']]
y = df['total_bill']

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=10)

In [11]:
X_train

,tip,sex,smoker,day,time,size
58,1.76,Male,Yes,Sat,Dinner,2
1,1.66,Male,No,Sun,Dinner,3
2,3.50,Male,No,Sun,Dinner,3
68,2.01,Male,No,Sat,Dinner,2
184,3.00,Male,Yes,Sun,Dinner,2
...,...,...,...,...,...,...
64,2.64,Male,No,Sat,Dinner,3
15,3.92,Male,No,Sun,Dinner,2
228,2.72,Male,No,Sat,Dinner,2
125,4.20,Female,No,Thur,Lunch,6


In [13]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 183 entries, 58 to 9
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   tip     183 non-null    float64 
 1   sex     183 non-null    category
 2   smoker  183 non-null    category
 3   day     183 non-null    category
 4   time    183 non-null    category
 5   size    183 non-null    int64   
dtypes: category(4), float64(1), int64(1)
memory usage: 5.6 KB


In [15]:
X_train["sex"].value_counts()

,count
sex,
Male,116
Female,67


In [16]:
X_train["day"].value_counts()

,count
day,
Sat,63
Sun,59
Thur,46
Fri,15


In [19]:
X_train["smoker"].value_counts()

,count
smoker,
No,118
Yes,65


In [17]:
X_train["time"].value_counts()

,count
time,
Dinner,133
Lunch,50


In [20]:
# Encoding (LabelEncoding and One HOt Encoding)
from sklearn.preprocessing import LabelEncoder
le1 = LabelEncoder()
le2 = LabelEncoder()
le3 = LabelEncoder()
X_train['sex'] = le1.fit_transform(X_train['sex'])
X_train['smoker'] = le2.fit_transform(X_train['smoker'])
X_train['time'] = le3.fit_transform(X_train['time'])
X_test['sex'] = le1.transform(X_test['sex'])
X_test['smoker'] = le2.transform(X_test['smoker'])
X_test['time'] = le3.transform(X_test['time'])

In [29]:
print(X_train['sex'].value_counts(), X_test['sex'].value_counts())

sex
1    116
0     67
Name: count, dtype: int64 sex
1    41
0    20
Name: count, dtype: int64


In [23]:
X_train['smoker'].value_counts()

,count
smoker,
0,118
1,65


In [31]:
X_train.head(3)

,tip,sex,smoker,day,time,size
58,1.76,1,1,Sat,0,2
1,1.66,1,0,Sun,0,3
2,3.50,1,0,Sun,0,3


In [33]:
# One Hot Encoding
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
ct = ColumnTransformer([('encoder', OneHotEncoder(drop='first'), [3])], remainder='passthrough')

In [36]:
import sys
np.printoptions(threshold=sys.maxsize)
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

In [40]:
pd.DataFrame(X_train).head()

,0,1,2,3,4,5,6,7
0,1.0,0.0,0.0,1.76,1.0,1.0,0.0,2.0
1,0.0,1.0,0.0,1.66,1.0,0.0,0.0,3.0
2,0.0,1.0,0.0,3.50,1.0,0.0,0.0,3.0
3,1.0,0.0,0.0,2.01,1.0,0.0,0.0,2.0
4,0.0,1.0,0.0,3.00,1.0,1.0,0.0,2.0


In [41]:
# SVR -- Support Vector Regression
from sklearn.svm import SVR
svr = SVR()


In [42]:
svr.fit(X_train,y_train)


SVR()

In [43]:
y_pred = svr.predict(X_test)

In [44]:
from sklearn.metrics import r2_score,mean_absolute_error
print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

0.46028114561159283
4.1486423210190235


In [45]:
from sklearn.model_selection import GridSearchCV
#defining parameter range
param_grid = {'C': [0.1, 1, 10, 100, 1000],
'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
'kernel': ['rbf']
}

In [46]:
grid = GridSearchCV(SVR(), param_grid, refit = True, verbose = 3,n_jobs=-1)
# fitting the model for grid search
grid.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


GridSearchCV(estimator=SVR(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
                         'kernel': ['rbf']},
             verbose=3)

In [47]:
grid.best_params_

{'C': 1000, 'gamma': 0.0001, 'kernel': 'rbf'}

In [49]:
y_pred1 = grid.predict(X_test)

In [50]:
from sklearn.metrics import r2_score,mean_absolute_error
print(r2_score(y_test,y_pred1))
print(mean_absolute_error(y_test,y_pred1))

0.5081618245078687
3.8685147526100234
